# components/file_browser_panel

> File browser panel configuration and rendering for browsing local audio/video files

In [ ]:
#| default_exp components.file_browser_panel

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
from typing import Any, Dict, List, Optional
from pathlib import Path

from cjm_fasthtml_file_browser.core.config import (
    FileBrowserConfig, FilterConfig, ViewConfig, SelectionMode, ViewMode, FileColumn
)
from cjm_fasthtml_file_browser.core.models import BrowserState, BrowserSelection
from cjm_fasthtml_file_browser.providers.local import LocalFileSystemProvider
from cjm_fasthtml_file_browser.components.browser import render_file_browser

from cjm_transcription_source_select.html_ids import SourceSelectHtmlIds

## Media Extensions

Audio and video extensions in dot-less format for `FilterConfig`.

In [ ]:
#| export
AUDIO_FILTER_EXTENSIONS = ["mp3", "wav", "flac", "aac", "ogg", "m4a", "wma", "opus"]
VIDEO_FILTER_EXTENSIONS = ["mp4", "mkv", "avi", "mov", "webm", "wmv", "flv"]
MEDIA_FILTER_EXTENSIONS = AUDIO_FILTER_EXTENSIONS + VIDEO_FILTER_EXTENSIONS

## Browser Configuration

Creates a `FileBrowserConfig` for audio/video file selection. Uses the `cjm-fasthtml-file-browser` library's built-in selection UI.

In [ ]:
#| export
def create_media_browser_config() -> FileBrowserConfig:  # Configured for audio/video file selection
    """Create file browser config for audio/video file selection."""
    return FileBrowserConfig(
        selection_mode=SelectionMode.MULTIPLE,
        selectable_types="files",
        view=ViewConfig(
            default_mode=ViewMode.LIST,
            allow_mode_toggle=False,
            columns=[FileColumn.NAME, FileColumn.SIZE, FileColumn.MODIFIED],
            sort_folders_first=True,
        ),
        filter=FilterConfig(
            allowed_extensions=MEDIA_FILTER_EXTENSIONS,
            show_directories=True,
            show_hidden=False,
        ),
        show_path_bar=True,
        show_breadcrumbs=True,
        show_toolbar=False,
        show_parent_navigation=True,
        container_id=SourceSelectHtmlIds.BROWSER_PANEL,
        content_id=SourceSelectHtmlIds.BROWSER_FILE_LIST,
    )

## Browser State

Helpers to create and restore `BrowserState` from step state.

In [ ]:
#| export
def get_browser_state(
    step_state: Dict[str, Any],  # Current step state
    default_path: str = "",  # Default directory if no state exists
) -> BrowserState:  # Browser state for the file browser
    """Get or create BrowserState from step state."""
    state_dict = step_state.get("browser_state", {})
    if state_dict:
        return BrowserState.from_dict(state_dict)
    return BrowserState(current_path=default_path or str(Path.home()))

In [ ]:
#| export
def sync_browser_selection(
    browser_state: BrowserState,  # Browser state to update
    selected_files: List[Dict[str, Any]],  # Current selected_files from step state
) -> None:  # Modifies browser_state in place
    """Sync browser selection state with the selected_files list."""
    browser_state.selection.selected_paths = [f["path"] for f in selected_files]

## Render Browser Panel

Thin wrapper around `render_file_browser()` that handles provider listing and HTMX targeting.

In [ ]:
#| export
def render_browser_panel(
    browser_state: BrowserState,  # Current browser state
    config: FileBrowserConfig,  # Browser configuration
    provider: LocalFileSystemProvider,  # File system provider
    navigate_url: str,  # URL for directory navigation
    select_url: str,  # URL for file selection toggle
    home_path: str = "",  # Home directory for nav buttons
) -> Any:  # Rendered file browser component
    """Render the file browser panel using the library's built-in UI."""
    listing = provider.list_directory(browser_state.current_path)
    hx_target = SourceSelectHtmlIds.as_selector(SourceSelectHtmlIds.BROWSER_PANEL)

    return render_file_browser(
        listing=listing,
        config=config,
        state=browser_state,
        navigate_url=navigate_url,
        select_url=select_url,
        toggle_view_url="",
        change_sort_url="",
        refresh_url=navigate_url,
        path_input_url=navigate_url,
        home_path=home_path or str(Path.home()),
        hx_target=hx_target,
    )

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()